In [1]:
import pandas as pd

def detect_columns(df):
    numerical_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    return numerical_cols, categorical_cols


In [2]:
import re

def detect_categorical_type(df, col):
    """
    Detect whether a categorical column is ordinal or nominal
    using semantic patterns common in Kaggle & real-world datasets.
    """

    values = (
        df[col]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .unique()
    )

    # ---------- ORDINAL WORD GROUPS ----------
    ordinal_sets = [
        {"low", "medium", "high"},
        {"very low", "low", "medium", "high", "very high"},
        {"poor", "average", "good", "excellent"},
        {"beginner", "intermediate", "advanced", "expert"},
        {"junior", "mid", "senior", "lead"},
        {"small", "medium", "large", "extra large"},
        {"bronze", "silver", "gold", "platinum"},
        {"yes", "no"}  # binary but ordered in many datasets
    ]

    # ---------- NOMINAL ENTITY WORDS ----------
    nominal_entities = {
        "male", "female",
        "city", "state", "country", "region",
        "department", "team",
        "type", "category",
        "brand", "product", "model",
        "color",
        "occupation", "job", "role",
        "company", "organization"
    }

    # ---------- RULE 1: Explicit nominal entities ----------
    if any(v in nominal_entities for v in values):
        return "nominal"

    # ---------- RULE 2: Ordinal keyword matching ----------
    for ord_set in ordinal_sets:
        if set(values).issubset(ord_set):
            return "ordinal"

    # ---------- RULE 3: Numeric ranking ----------
    if all(v.isdigit() for v in values):
        return "ordinal"

    # ---------- RULE 4: Level pattern (Level 1, Level 2...) ----------
    if all(re.match(r"(level|lvl)\s*\d+", v) for v in values):
        return "ordinal"

    # ---------- RULE 5: Grade pattern ----------
    if set(values).issubset(set("abcdef")):
        return "ordinal"

    # ---------- SAFE DEFAULT ----------
    return "nominal"


In [12]:
import pandas as pd

df = pd.read_csv("/content/customer (1).csv")

In [15]:
df

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No
5,31,Female,Average,School,Yes
6,18,Male,Good,School,No
7,60,Female,Poor,School,Yes
8,65,Female,Average,UG,No
9,74,Male,Good,UG,Yes


In [13]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()


In [14]:
num_cols

['age']

In [22]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()


In [23]:
cat_cols

['gender', 'review', 'education', 'purchased']

In [33]:
import re

def detect_categorical_type(df, col):
    """
    High-accuracy ordinal vs nominal detector
    Optimized for Kaggle, education, job, company, position datasets.
    """

    # -------------------- PREPROCESS --------------------
    values = (
        df[col]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
        .unique()
    )

    col_name = col.lower()

    # -------------------- STRONG NOMINAL COLUMN HINTS --------------------
    # If column name itself indicates nominal → STOP
    nominal_col_keywords = {
        "id", "name",
        "gender", "sex",
        "city", "state", "country", "region",
        "department", "team",
        "category", "type",
        "brand", "product", "model",
        "color",
        "company", "organization", "employer",
        "industry"
    }

    if any(keyword in col_name for keyword in nominal_col_keywords):
        return "nominal"

    # -------------------- EDUCATION (ORDINAL) --------------------
    education_levels = {
        "primary", "secondary",
        "high school", "higher secondary",
        "diploma",
        "associate",
        "bachelor", "undergraduate",
        "master", "postgraduate",
        "phd", "doctorate"
    }

    if set(values).issubset(education_levels):
        return "ordinal"

    # -------------------- JOB / POSITION LEVELS (ORDINAL) --------------------
    job_levels = {
        "intern",
        "junior",
        "associate",
        "mid",
        "senior",
        "lead",
        "manager",
        "director",
        "vp",
        "vice president",
        "c-level",
        "ceo", "cto", "cfo"
    }

    if set(values).issubset(job_levels):
        return "ordinal"

    # -------------------- EXPERIENCE LEVELS --------------------
    experience_levels = {
        "fresher",
        "entry",
        "beginner",
        "intermediate",
        "experienced",
        "advanced",
        "expert"
    }

    if set(values).issubset(experience_levels):
        return "ordinal"

    # -------------------- RATING / PERFORMANCE --------------------
    rating_levels = {
        "very poor",
        "poor",
        "average",
        "good",
        "very good",
        "excellent",
        "outstanding"
    }

    if set(values).issubset(rating_levels):
        return "ordinal"

    # -------------------- PRIORITY / SEVERITY --------------------
    priority_levels = {
        "low",
        "medium",
        "high",
        "critical",
        "blocker"
    }

    if set(values).issubset(priority_levels):
        return "ordinal"

    # -------------------- GRADE PATTERN (A, B, C...) --------------------
    if set(values).issubset({"a", "b", "c", "d", "e", "f"}):
        return "ordinal"

    # -------------------- NUMERIC ORDINAL --------------------
    # Example: ratings 1–5, levels 1–10
    if all(v.isdigit() for v in values):
        return "ordinal"

    # -------------------- LEVEL / BAND / TIER PATTERN --------------------
    if all(re.match(r"(level|lvl|band|tier|stage)\s*\d+", v) for v in values):
        return "ordinal"

    # -------------------- SAFE DEFAULT --------------------
    return "nominal"


In [32]:
detect_categorical_type(df,'gender')

'nominal'

In [8]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression())
])


In [11]:
pipeline.fit(X_train, y_train)


NameError: name 'X_train' is not defined